<a href="https://colab.research.google.com/github/BrenoLuna861/Projects-AI-Talent-Academy/blob/main/Aula_04_Alunos_Pr%C3%A1tica_02_Semana_2_AI_Talent_Academy_Grupo_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 04 – Prática 02 – Semana 2
## AI Talent Academy — Grupo 01

**Integrantes:**

- Breno Luna
- Ricardo Lima
- Paula Carlesso

---

**Conteúdo desta entrega:**

1. Demonstração de tokenização com `tiktoken`
2. Experimentos 1 e 2 (análise de sentimento e agrupamento em tópicos)
3. Exercício: função `classificar_reclamacao` implementada
4. Teste individual e batch de 5 casos com cálculo de acurácia
5. Respostas das Perguntas de Reflexão (ao final do notebook)

In [ ]:
# Baixando as duas bibliotecas que serão utilizadas e que não vem por padrão no Google Colab
!pip install datasets tiktoken -q

In [50]:
def demonstrar_tokens(texto, model="gpt-5"):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(texto)
    print(f"Texto: {texto}")
    print(f"Lista de IDs de Tokens: {tokens}")
    print(f"Quantidade de tokens: {len(tokens)}")

    # Decodificando cada token individualmente para mostrar a divisão
    print("Divisão visual:", [encoding.decode([t]) for t in tokens])

# Rodando com prompts meus para ver como o modelo enxerga o texto:

# 1) Uma frase do dominio do exercicio
demonstrar_tokens("Classifique esta reclamacao em uma das categorias.")

# 2) Acento custa token: mesma palavra com e sem
demonstrar_tokens("cobranca")
demonstrar_tokens("cobranca indevida")

# 3) O espaco faz parte do token seguinte
demonstrar_tokens("sinal")
demonstrar_tokens(" sinal")

Texto: Classifique esta reclamacao em uma das categorias.
Lista de IDs de Tokens: [2579, 29455, 6476, 79845, 16693, 863, 3030, 2331, 104206, 13]
Quantidade de tokens: 10
Divisão visual: ['Class', 'ifique', ' esta', ' reclam', 'acao', ' em', ' uma', ' das', ' categorias', '.']
Texto: cobranca
Lista de IDs de Tokens: [66, 58774, 19863]
Quantidade de tokens: 3
Divisão visual: ['c', 'obr', 'anca']
Texto: cobranca indevida
Lista de IDs de Tokens: [66, 58774, 19863, 6741, 32243]
Quantidade de tokens: 5
Divisão visual: ['c', 'obr', 'anca', ' inde', 'vida']
Texto: sinal
Lista de IDs de Tokens: [82, 1028]
Quantidade de tokens: 2
Divisão visual: ['s', 'inal']
Texto:  sinal
Lista de IDs de Tokens: [68376]
Quantidade de tokens: 1
Divisão visual: [' sinal']


## Sobre o dataset de exemplo (B2W-Reviews01):
O B2W-Reviews01 é um corpus aberto de avaliações de produtos. Ele contém mais de 130 mil avaliações coletadas em 2018 de clientes de comércio eletrônico, coletadas de sites como Americanas e outros e-commerces.

O B2W-Reviews01 oferece informações também sobre o perfil dos avaliadores, como gênero, idade e localização geográfica. O corpus também apresenta dois tipos diferentes de avaliações

In [ ]:
import tiktoken
from datasets import load_dataset

# Baixa e já carrega em memória, cacheado localmente
dataset = load_dataset("ruanchaves/b2w-reviews01", revision="refs/convert/parquet")

# Ver a estrutura
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['submission_date', 'reviewer_id', 'product_id', 'product_name', 'product_brand', 'site_category_lv1', 'site_category_lv2', 'review_title', 'overall_rating', 'recommend_to_a_friend', 'review_text', 'reviewer_birth_year', 'reviewer_gender', 'reviewer_state'],
        num_rows: 132373
    })
})


In [ ]:
# Acessar o split de treino, por exemplo
dados = dataset["train"]
colunas_para_manter = ['submission_date', 'product_id', 'product_name', 'review_title', 'review_text', 'reviewer_birth_year', 'reviewer_gender', 'reviewer_state']

# Converter pra pandas
df = dados.to_pandas()
df = df[colunas_para_manter]
df.head(1)

,submission_date,product_id,product_name,review_title,review_text,reviewer_birth_year,reviewer_gender,reviewer_state
0,2018-01-01 00:11:28,132532965,Notebook Asus Vivobook Max X541NA-GO472T Intel...,Bom,Estou contente com a compra entrega rápida o ú...,1958.0,F,RJ


### Experimento 1: Análise de Sentimento com LLM

Nesta tarefa, vamos pedir para a LLM analisar o texto da avaliação e classificar como **Positivo**, **Negativo** ou **Neutro**. É um ótimo momento para mostrar como o `system_prompt` ajuda a restringir a saída do modelo.

In [ ]:
# Configurando o cliente para realizar as chamadas para a API da OpenAI

from openai import OpenAI
from google.colab import userdata

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = userdata.get('NVD_API_KEY') # Adicionar o seu secret com esse nome!
)

In [ ]:
def analisar_sentimento_llm(texto):
    # Prompt de sistema definindo o comportamento da LLM
    system_prompt = """
        **PERSONA**: Você é um analista de sentimentos especializado em e-commerce.
        **CONTEXTO**: Nossa empresa é um e-commerce que recebe muitas revisões de nossos consumidores.
        **OBJETIVOS**: Responda apenas com UMA palavra: Positivo, Negativo ou Neutro. Não responda com nada mais.
    """

    # Chamada da API (usando o client já configurado anteriormente)
    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Analise o sentimento desta avaliação: {texto}"}
        ],
        temperature=0.1,
        extra_body={"chat_template_kwargs":{"enable_thinking":True}}
    )
    return response.choices[0].message.content.strip()

# Teste com uma amostra
exemplo_review = df.sample(1).iloc[0]['review_text']
print(f"Review: {exemplo_review}")
print(f"Sentimento Previsto: {analisar_sentimento_llm(exemplo_review)}")

Review: Será que é original? O valor está bem abaixo do preço normal. :/
Sentimento Previsto: Here's a thinking process:

1.  **Analyze User Input:**
   - the user is asking me to analyze the user input, which contains Chinese characters. Let me: "分析此人工具分析情感：Positivo, Negativo ou Neutro. Não responda com nada mais.


### Experimento 2: Agrupamento em Tópicos (Topic Tagging)

Aqui, o objetivo é mostrar como a LLM pode identificar o *assunto principal* (ex: Logística, Qualidade do Produto, Preço) para podermos agrupar comentários similares depois.

In [ ]:
def identificar_topico(texto):
    # Definimos categorias para facilitar o agrupamento posterior
    topicos_sugeridos = "Logística/Entrega, Qualidade do Produto, Preço/Custo-benefício, Atendimento, Usabilidade"

    system_prompt = f"""
        Identifique qual destes tópicos melhor descreve a reclamação: {topicos_sugeridos}. Responda apenas o nome do tópico.
        Só responda com os tópicos listados anteriormente. Sempre responda em português brasileiro. Não invente nenhum tópico diferente.
    """

    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": texto}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()

# Demonstração
amostra_topico = df.sample(1)
for i, row in amostra_topico.iterrows():
    topico = identificar_topico(row['review_text'])
    print(f"Texto: {row['review_text'][:110]}...")
    print(f"-> Tópico Identificado: {topico}\n")

Texto: Aparelho muito funcional até dar defeito 5 dias após a entrega. 3 usos apenas. Desapontada...
-> Tópico Identificado: Qualidade do Produto



# Exercício:
Nesta aula, aprenderemos a utilizar uma LLM via API para automatizar a classificação de problemas reportados por usuários.

### 1. Entendendo a Estrutura da Chamada
Uma chamada de LLM geralmente envolve:
*   **System Prompt**: Define o comportamento do modelo.
*   **User Prompt**: A entrada de dados ou pergunta.
*   **Temperature**: Controla a criatividade vs. precisão.

---

*Os blocos que necessitam de edição estarão com a flag 'ESCREVA AQUI'*

### Importando os dados:
---
Nesta etapa, estamos importando um conjunto de dados de reclamações de e-commerce. O arquivo contém duas abas principais:
1.  **dataset_treino**: Contém o histórico de reclamações com colunas como `title` (título da reclamação), `description` (relato detalhado do cliente).
2.  **problemas**: Uma lista de referência com todas as categorias de problemas mapeadas pela empresa.

Interpretamos esses dados comparando o que o cliente escreveu com a categoria atribuída manualmente, o que nos permite validar se uma Inteligência Artificial consegue realizar essa classificação de forma autônoma e precisa.

In [ ]:
# NÃO EDITE ESSA CÉLULA:
import pandas as pd

# O ID do arquivo extraído do link fornecido
file_id = '1_9MN5HDaESKDSa4CFNpTvqnOwLWPJJds'
direct_link = f'https://drive.google.com/uc?export=download&id={file_id}'

try:
    df = pd.read_excel(direct_link, sheet_name="dataset_treino")
    categorias = pd.read_excel(direct_link, sheet_name="problemas")
    print("Arquivos carregados com sucesso!")
except Exception as e:
    print(f"Erro ao ler o arquivo: {e}")

Arquivos carregados com sucesso!


In [ ]:
# Implemente a lógica de chamada da API

import re
import unicodedata


def _limpar_resposta(texto):
    """Tira lixo que o modelo às vezes gruda na resposta."""
    # o nemotron às vezes devolve o raciocínio mesmo com thinking desligado
    texto = re.sub(r"<think>.*?</think>", "", texto, flags=re.DOTALL)
    return texto.strip().strip('."\'*').strip()


def classificar_reclamacao(titulo, descricao):
    # 1. Coloque nessa lista todas as categorias:
    # Leio da aba "problemas" em vez de digitar na mão: assim não erro acento
    # nem maiúscula (o gabarito é comparado string com string), e se a empresa
    # criar uma categoria nova o código continua certo sem eu mexer em nada.
    coluna = categorias.columns[0]
    CATEGORIAS = sorted(
        categorias[coluna].dropna().astype(str).str.strip().unique().tolist()
    )
    lista_formatada = "\n".join(f"- {c}" for c in CATEGORIAS)

    # Obs.: rodei a célula acima e olhei os dados antes de escrever o prompt.
    # A aba "problemas" tem 14 categorias na coluna 'problem', e o domínio
    # é telefonia (sinal, chip, recarga, portabilidade) - não e-commerce.
    # Ajustei a PERSONA pra isso.

    # 2. Defina aqui o seu System Prompt
    # Dica: Diga ao modelo quem ele é e quais as regras para a resposta.
    # Segui a estrutura dos Experimentos 1 e 2: PERSONA + CONTEXTO + REGRAS.
    # O que mais importa é colar a lista literal aqui dentro - é o que segura
    # a alucinação de categoria: o modelo não pode copiar o que não vê.
    SYSTEM_PROMPT = f"""
        **PERSONA**: Você é um analista de Customer Experience (CX) de uma operadora de telecom,
        especializado em triagem de reclamações de clientes.

        **CONTEXTO**: Toda reclamação que chega precisa ser encaixada em exatamente uma
        das categorias já mapeadas pela empresa. Essa categoria alimenta os relatórios
        internos do time, então o nome precisa sair escrito exatamente como está na lista.

        **CATEGORIAS PERMITIDAS**:
{lista_formatada}

        **REGRAS**:
        1. Responda com o nome de UMA única categoria, copiada literalmente da lista acima.
        2. Não invente categorias novas, não traduza e não reescreva os nomes.
        3. Não escreva explicação, justificativa, pontuação final ou qualquer outro texto.
        4. Se a reclamação parecer caber em mais de uma categoria, escolha a que descreve
           o problema PRINCIPAL relatado pelo cliente.
        5. Sempre responda em português brasileiro.
        6. O título é um resumo curto escrito pelo próprio cliente e às vezes já coincide
           com o nome de uma categoria; considere título e descrição juntos, nunca só o título.
    """

    # 3. Monte o prompt do usuário
    # Rotulei os dois campos. Sem rótulo o modelo mistura título com descrição
    # e às vezes trata o título como se fosse instrução minha.
    USER_PROMPT = f"""Classifique a reclamação abaixo.

Título: {titulo}
Descrição: {descricao}"""

    # 4. Modelo NVIDIA utilizado:
    MODELO = "nvidia/nemotron-3.5-lightning-30b-a3b" # [Pode editar caso esteja utilizando outro modelo]

    # 5. Temperatura: quão criativo o modelo precisa ser para essa tarefa?
    # 0 (zero). Isso não é tarefa criativa, é classificação em lista fechada.
    # Quero que a mesma reclamação devolva sempre a mesma categoria - senão eu
    # nem consigo medir acurácia direito, porque o resultado muda a cada run.
    TEMPERATURA = 0

    # 6. O modo de reasoning (pensamento estendido) precisa estar ativado?
    # False. A decisão é quase um "de-para" entre o texto e uma lista de rótulos.
    # Ligar reasoning aqui só gasta token, aumenta a latência e ainda arrisca o
    # modelo devolver o raciocínio junto, quebrando a regra de "só a categoria".
    THINKING = False

    # 7. Realize a chamada ao client.chat.completions.create()
    #Lembre-se de passar o SYSTEM_PROMPT e o USER_PROMPT
    resposta = client.chat.completions.create(
        model= MODELO,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ],
        temperature=TEMPERATURA,
        extra_body={"chat_template_kwargs":{"enable_thinking":THINKING}},
        stream=False
    )

    return _limpar_resposta(resposta.choices[0].message.content)

### 2. Teste Prático
Vamos selecionar uma linha aleatória do nosso dataset e ver se o modelo consegue prever a coluna `problem` corretamente.

In [ ]:
# Selecionando um exemplo aleatório para teste
exemplo = df.sample(1).iloc[0]

print(f"--- DADOS DE ENTRADA ---")
print(f"Título: {exemplo['title']}")
print(f"Descrição: {exemplo['description'][:200]}...")

# Chamada da LLM
predicao = classificar_reclamacao(exemplo['title'], exemplo['description'])

print(f"\n--- RESULTADO DA LLM ---")
print(f"Categoria Prevista: {predicao}")

--- DADOS DE ENTRADA ---
Título: Crédito não entrou ..
Descrição: Prezados fiz uma recarga de 40,00 pelo banco Next em 30/11/2022 e hoje 01/12/2022 a recarga ainda não entrou para validar meus benefícios . E através desse número que trabalho , faço minhas ligações e...

--- RESULTADO DA LLM ---
Categoria Prevista: Consumo de crédito


## Perguntas de Reflexão e Validação
---

### 1. Desempenho quantitativo
Execute a célula de teste abaixo para 5 exemplos aleatórios:
- **a)** Como a decisão da IA se comparou ao gabarito original?
- **b)** Em números absolutos, quantos acertos e erros ocorreram?
- **c)** Qual foi a acurácia (porcentagem de acerto) observada?

### 2. Análise qualitativa e confiabilidade
- **a)** Nos casos de erro, a resposta da IA ainda faz sentido? O erro foi por ambiguidade do texto ou falha do modelo?
- **b)** O modelo 'inventou' alguma categoria que não estava na lista original (alucinação)?
- **c)** Você confiaria em automatizar 100% desse processo ou manteria uma revisão humana?

### 3. Impacto de negócio e próximos passos
- **a)** Como essa classificação automática poderia ajudar o time de Customer Experience (CX)?
- **b)** O que acontece se mudarmos a `temperatura` para 0.0? E se aumentarmos para 1.0?
- **c)** Você sente que se houvesse mais alguma feature (coluna do dataset), isso poderia ajudar o modelo a ser mais preciso? Se sim, qual feature você imagina?

In [ ]:
# Dataset com a label verdadeira:
try:
    df_full = pd.read_excel(direct_link, sheet_name="full_dataset")
    categorias = pd.read_excel(direct_link, sheet_name="problemas")
    print("Arquivos carregados com sucesso!")
except Exception as e:
    print(f"Erro ao ler o arquivo: {e}")

Arquivos carregados com sucesso!


In [ ]:
# Código para responder à Pergunta 1 (Batch de 5 testes)
amostra_teste = df_full.sample(5)
acertos = 0

print(f"{'Item':<5} | {'Real':<25} | {'Previsto':<25} | {'Resultado'}")
print("-" * 80)

for i, (idx, row) in enumerate(amostra_teste.iterrows()):
    real = row['problem']
    previsto = classificar_reclamacao(row['title'], row['description'])

    status = "✅" if real.strip().lower() == previsto.strip().lower() else "❌"
    if status == "✅": acertos += 1

    print(f"{i+1:<5} | {real[:25]:<25} | {previsto[:25]:<25} | {status}")

print("-" * 80)
print(f"Total de Acertos: {acertos} de 5")
print(f"Acurácia: {(acertos/5)*100}%")

Item  | Real                      | Previsto                  | Resultado
--------------------------------------------------------------------------------
1     | Qualidade da internet     | Instabilidade do sinal    | ❌
2     | Mudança de plano          | Mudança de plano          | ✅
3     | Cancelamento              | Cancelamento              | ✅
4     | Promoções                 | Planos e tarifas          | ❌
5     | Consumo de crédito        | Planos e tarifas          | ❌
--------------------------------------------------------------------------------
Total de Acertos: 2 de 5
Acurácia: 40.0%


# Minhas respostas às Perguntas de Reflexão

## 1. Desempenho quantitativo

**a) Como a decisão da IA se comparou ao gabarito original?**

| # | Gabarito (real) | Previsto pela LLM | Acertou? |
|---|---|---|---|
| 1 | Qualidade da internet | Instabilidade do sinal | ❌ |
| 2 | Mudança de plano | Mudança de plano | ✅ |
| 3 | Cancelamento | Cancelamento | ✅ |
| 4 | Promoções | Planos e tarifas | ❌ |
| 5 | Consumo de crédito | Planos e tarifas | ❌ |

**b)** 2 acertos e 3 erros de 5.

**c) Acurácia: 40,0%.**

Uma ressalva que eu quero registrar, e que eu mesmo comprovei: rodei essa célula duas
vezes, **sem mudar uma linha de código**. Na primeira execução deu 1/5 (20%) e na segunda
2/5 (40%). O modelo não mudou — mudou a amostra sorteada.

Com 5 exemplos, cada item vale 20 pontos percentuais, então esse número não mede nada com
segurança. Para confiar na acurácia eu rodaria 50–100 casos e olharia o resultado **por
categoria**, não só a média: o modelo pode ir bem nas categorias comuns e mal justamente
nas raras, e a média esconde isso.

## 2. Análise qualitativa e confiabilidade

**a) Nos casos de erro, a resposta ainda faz sentido? Ambiguidade ou falha do modelo?**

Faz sentido, sim — e esse é o achado mais interessante da atividade. **Os erros não são
chutes aleatórios: são pares de categorias vizinhas.**

- *Qualidade da internet* ↔ *Instabilidade do sinal*
- *Promoções* ↔ *Planos e tarifas*
- *Consumo de crédito* ↔ *Planos e tarifas*

Em todos os três, as duas categorias descrevem situações que se sobrepõem. A fronteira
entre elas é uma **regra de negócio que mora na cabeça do time de CX e não está escrita em
lugar nenhum do prompt**. Se dois humanos rotulassem essas mesmas reclamações, eles também
discordariam entre si.

Ou seja: em boa parte dos casos não é falha do modelo, é ambiguidade do rótulo. Isso muda o
que fazer a seguir — o caminho é **escrever melhor o prompt** (dar critério de desempate e
exemplos), não trocar de modelo.

**b) O modelo inventou alguma categoria fora da lista (alucinação)?**

**Não. Zero alucinações.** As 5 respostas são categorias que existem literalmente na aba
`problemas`. Isso não foi sorte — foi consequência de três decisões no System Prompt:

1. colei a **lista literal** das 14 categorias dentro do prompt (o modelo não pode copiar o
   que não vê);
2. mandei explicitamente *copie literalmente, não traduza, não reescreva*;
3. proibi explicação, justificativa e pontuação final.

Também li as categorias direto de `categorias[coluna]` em vez de digitar na mão — assim não
corro o risco de errar um acento e transformar acerto em erro na comparação.

**c) Confiaria em automatizar 100% do processo?**

Não. E o motivo não é desconfiança do modelo, é a **distribuição do erro**: mesmo com uma
acurácia alta, a fração que erra vai para a fila errada, e reclamação mal roteada é cliente
esperando mais tempo.

O que eu faria é um **fluxo híbrido por confiança**: pedir ao modelo a categoria e um nível
de confiança; caso confiante, classifica sozinho; caso de baixa confiança, resposta fora da
lista, ou texto que menciona mais de um problema, vai para revisão humana. Isso tira o
trabalho repetitivo do time sem entregar o caso duvidoso para a máquina — e as revisões
humanas viram dado novo para melhorar o prompt.

## 3. Impacto de negócio e próximos passos

**a) Como isso ajudaria o time de CX?**

- **Roteamento na entrada:** a reclamação já chega no time certo (cobrança, rede, retenção)
  sem triagem manual, reduzindo o tempo até a primeira resposta.
- **Visão agregada:** com tudo categorizado dá para montar um painel de reclamações por
  categoria por semana. Um pico súbito em *Instabilidade do sinal* denuncia um problema de
  rede em alguma região — hoje isso só aparece quando alguém lê os relatos um por um.
- **Priorização:** cruzando a categoria com valor do plano ou recorrência do cliente, dá
  para atacar primeiro o que dói mais.

O valor não está na classificação em si, e sim no que ela **destrava**: sem rótulo, milhares
de reclamações são texto solto; com rótulo, viram uma tabela que o negócio consegue ler.

**b) O que acontece mudando a temperatura para 0.0 e para 1.0?**

A temperatura controla o quanto o modelo se afasta do token mais provável ao gerar.

- **0.0** → praticamente determinístico. A mesma reclamação devolve sempre a mesma
  categoria. É o que eu quero aqui: reproduzível e auditável, e a acurácia medida hoje
  continua valendo amanhã. Foi por isso que usei `TEMPERATURA = 0`.
- **1.0** → o modelo passa a amostrar entre alternativas plausíveis. Num caso ambíguo ele
  devolveria *Planos e tarifas* numa execução e *Promoções* na seguinte, **com a mesma
  entrada**. Também sobe a chance de ele enfeitar a resposta — escrever uma frase em vez da
  categoria seca — ou inventar um rótulo novo.

Criatividade é ótima para redigir a resposta ao cliente. É ruim para escolher o rótulo.

**c) Alguma feature (coluna) a mais ajudaria o modelo?**

Sim, em ordem de retorno esperado:

1. **Status da linha / do pedido** (ativa, suspensa, em portabilidade, bloqueada) —
   desambiguaria sozinha boa parte da família de categorias de linha e sinal, que é
   justamente onde os erros se concentraram.
2. **Valor da fatura e histórico de cobrança** — separaria *Cobrança indevida* de
   *Planos e tarifas*, outro par que confundiu.
3. **Tempo entre a contratação e a reclamação** — reclamação nos primeiros dias tende a ser
   chip/ativação; meses depois tende a ser cobrança ou qualidade.
4. **Região / antena** — ajudaria a separar problema individual de falha de rede local.

Mas antes de adicionar coluna eu tentaria algo mais barato: **few-shot**. Colocar 3 a 5
exemplos já resolvidos dentro do prompt, escolhidos de propósito nos pares que confundiram
(*Qualidade da internet* vs *Instabilidade do sinal*, *Promoções* vs *Planos e tarifas*).
Isso ensina a fronteira entre as categorias sem precisar de dado novo e sem custo de
engenharia.

---

## O que eu levo dessa aula

- O System Prompt não é enfeite: colar a lista de categorias dentro dele é o que separou uma
  saída utilizável de uma bagunça de texto livre — deu zero alucinação.
- Temperatura é escolha de engenharia, não gosto pessoal: casa com o tipo de tarefa.
- **Reasoning nem sempre ajuda.** No Experimento 1, com `enable_thinking: True`, o modelo
  devolveu o raciocínio inteiro no lugar da palavra única. Por isso usei `THINKING = False`
  no exercício: a tarefa é um "de-para" entre texto e uma lista fechada de rótulos.
- **Como eu comparo o resultado importa tanto quanto o resultado**: uma comparação exata
  demais transformaria acerto em erro por causa de um acento ou ponto final.
- 5 amostras dão intuição, não métrica — e eu vi isso na prática, com 20% e 40% no mesmo
  código.